# Inhalte aus Metadaten extrahieren

### Benötigte Bibliotheken importieren:

In [ ]:
from bs4 import BeautifulSoup as soup
import unicodedata
from lxml import etree
import pandas as pd

### Laden des XML-Files: 

In [ ]:
# Laden der MARC-xml-Datei in ElementTree und umwandeln in eine Liste an Datensätzen: 
tree = etree.parse('20260526_MARC21-xml_comic_and_hsg=741.5_and_mat=online.xml')
root = tree.getroot()
ns = {'marc': 'http://www.loc.gov/MARC21/slim'}
records = root.findall('.//marc:record', namespaces=ns)
print("Gefundene Records:", len(records))

### Erstellen einer Funktion für die Extraktion der gewünschten Informationen

Hier können die Felder entsprechend den eigenen Wünschen angepasst werden:

In [ ]:
def parse_record(record):
    
    ns = {"marc": "http://www.loc.gov/MARC21/slim"} 
    
    def extract_text(xpath_query):
        fields = record.xpath(xpath_query, namespaces=ns)
        if fields:
            return "; ".join(field.text for field in fields if field.text)
        return None

    # Hier Felder anpassen: 
    # 1. Vor dem Gleichheitszeichen steht eine frei wählbare Bezeichnung der Information (es wird eine Variable erzeugt, bspw. mit dem Name "titel")
    # 2. Nach dem Gleichheitszeichen wird die oben angelegte Funktion zur Extraktion des Feldinhalts aus dem jeweiligen MARC12-xml-Feld aufgerufen. Hierfür 
    #        muss lediglich der Typ des Feldes (datafield oder controlfield), die Nummer des Feldes ("tag") und, wenn zutreffend, das zugehörige Unterfeld (subfield)
    #        angepasst werden. 
    
    idn = extract_text("marc:controlfield[@tag='001']")
    author = extract_text("marc:datafield[@tag='100']/marc:subfield[@code='a']")
    author2 = extract_text("marc:datafield[@tag='700']/marc:subfield[@code='a']")
    title = extract_text("marc:datafield[@tag='245']/marc:subfield[@code='a']")
    subtitle = extract_text("marc:datafield[@tag='245']/marc:subfield[@code='b']")
    statement_of_responsibility = extract_text("marc:datafield[@tag='245']/marc:subfield[@code='c']")
    publisher = extract_text("marc:datafield[@tag='264']/marc:subfield[@code='b']")
    place = extract_text("marc:datafield[@tag='264']/marc:subfield[@code='a']")
    pages = extract_text("marc:datafield[@tag='300']/marc:subfield[@code='a']")
    pub_year = extract_text("marc:datafield[@tag='264']/marc:subfield[@code='c']")
    #year_008 = extract_text("marc:controlfield[@tag='008']")[7:11]
    links = extract_text("marc:datafield[@tag='856']/marc:subfield[@code='u']")
    ddc = extract_text("marc:datafield[@tag='082']/marc:subfield[@code='a']")
    
    # 3. Zuletzt müssen neue Variablen hier ergänzt, entfernte gelöscht werden.
    #     Zum Löschen dafür jeweils den gesamten Teil zwischen zwei Kommata löschen, also bsw. "links":links um die Ausgabe von Links zu entfernen
    #     Zum Ergänzen einfach an gewünschter Stelle nach vorliegendem Schema neue Variablen ergänzen: Hier ist der erste Teil in "" wieder ein frei wählbarer
    #     Name (dies wird der Spaltenname in einer später auszugebenden Tabelle), nach dem Doppelpunkt folgt dann der zugehörigen eben oben definierte Variablenname:
    return {"idn": idn, "author": author, "author2": author2,"title": title, "subtitle": subtitle, "responsible":statement_of_responsibility, 
            "publisher": publisher, "place": place, "pages": pages, "pub_year":pub_year, "links":links, "ddc": ddc}

Aufrufen der Funktion, Übergabe der einzelnen Datensätze und Transformation in eine Tabelle: 

In [ ]:
result = [parse_record(record) for record in records]
df = pd.DataFrame(result)
df

### Speichern

In [ ]:
df.to_csv("Comics_digital.csv", index=False, encoding="utf-8")